# CommodityOS - Global Market Intelligence System
## 2400+ CrawlSpiders | 9 Regions | 180+ Countries

**How to use:**
1. Upload `CommodityOS_v1.0.zip` to your Google Drive root
2. Run all cells (Runtime → Run all)
3. Dashboard auto-published to GitHub Pages
4. Data saved to Google Drive automatically

## Step 1: Mount Google Drive & Install Dependencies

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted successfully!')

In [ ]:
# Install system dependencies
!apt-get update -qq && apt-get install -y -qq git > /dev/null 2>&1
print('System dependencies installed')

In [ ]:
# Install Python dependencies
!pip install -q networkx aiohttp psutil beautifulsoup4 lxml
print('Python dependencies installed')

## Step 2: Extract Project from Google Drive

In [ ]:
import os
import zipfile
import shutil

# Find and extract zip file from Google Drive
DRIVE_ZIP = '/content/drive/MyDrive/CommodityOS_v1.0.zip'
PROJECT_DIR = '/content/Commodity'

# Create project directory
os.makedirs(PROJECT_DIR, exist_ok=True)

if os.path.exists(DRIVE_ZIP):
    print(f'Found zip: {DRIVE_ZIP}')
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
        z.extractall(PROJECT_DIR)
    print(f'Extracted to: {PROJECT_DIR}')
else:
    # Try to find any Commodity zip
    for f in os.listdir('/content/drive/MyDrive/'):
        if 'commodity' in f.lower() and f.endswith('.zip'):
            DRIVE_ZIP = f'/content/drive/MyDrive/{f}'
            with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
                z.extractall(PROJECT_DIR)
            print(f'Found and extracted: {f}')
            break
    else:
        print('ERROR: No CommodityOS zip found in Google Drive root!')
        print('Please upload CommodityOS_v1.0.zip to your Google Drive root folder.')

In [ ]:
# Verify extraction and list project files
os.chdir(PROJECT_DIR)
print('Project files:')
for item in sorted(os.listdir('.')):
    if not item.startswith('.'):
        print(f'  {item}')

## Step 3: Configure Git & GitHub Pages

In [ ]:
# Configure git credentials
# EDIT THESE VALUES with your GitHub credentials
GITHUB_TOKEN = ''  # Your GitHub Personal Access Token
GITHUB_REPO = 'https://github.com/iamgokhu/Commodityapp01.git'
GITHUB_BRANCH = 'main'

# Configure git
!git config --global user.email "commodity@os.local"
!git config --global user.name "CommodityOS"

# Set up remote with token authentication
if GITHUB_TOKEN:
    repo_url = GITHUB_REPO.replace('https://', f'https://{GITHUB_TOKEN}@')
    !git remote set-url origin {repo_url}
    print('GitHub configured with token authentication')
else:
    !git remote set-url origin {GITHUB_REPO}
    print('WARNING: No GitHub token set. Push will fail without authentication.')
    print('Set GITHUB_TOKEN variable above with your Personal Access Token.')

## Step 4: Initialize 2400+ CrawlSpiders

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from commodity_os.crawlers.spider_factory import SpiderFactory

# Generate all spiders
factory = SpiderFactory(spiders_per_country=5, max_pages=5)
stats = factory.get_stats()

print(f'='*60)
print(f'SPIDER FACTORY RESULTS')
print(f'='*60)
print(f'Total Spiders: {stats["total_spiders"]}')
print(f'Regions: {stats["regions"]}')
print(f'Countries: {stats["countries"]}')
print(f'')
print('By Region:')
for region, count in sorted(stats['by_region'].items()):
    print(f'  {region:20s} {count:5d} spiders')
print(f'')
print('By Type:')
for stype, count in sorted(stats['by_type'].items()):
    print(f'  {stype:20s} {count:5d} spiders')
print(f'='*60)

## Step 5: Run Global Collection Pipeline

In [ ]:
import asyncio
import json
import time
from pathlib import Path

async def run_global_collection():
    """Run the full global collection pipeline."""
    print(f'='*60)
    print(f'STARTING GLOBAL COLLECTION')
    print(f'='*60)
    start_time = time.time()

    # Run all spiders in batches
    all_entities, stats = await factory.run_batch(batch_size=50, delay=3.0)

    elapsed = time.time() - start_time
    print(f'')
    print(f'COLLECTION COMPLETE')
    print(f'  Time: {elapsed:.1f}s')
    print(f'  Total entities: {len(all_entities)}')
    print(f'  Batches run: {stats["batches"]}')

    return all_entities, stats

# Run collection
entities, collection_stats = await run_global_collection()

## Step 6: Save Data to Google Drive

In [ ]:
import json
from pathlib import Path
from datetime import datetime

# Save entities to Google Drive
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/CommodityOS_Data')
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Save raw entities
entities_file = DRIVE_DATA_DIR / 'all_entities.json'
with open(entities_file, 'w', encoding='utf-8') as f:
    json.dump(entities, f, indent=2, ensure_ascii=False)
print(f'Saved {len(entities)} entities to Google Drive')

# Save collection stats
stats_file = DRIVE_DATA_DIR / 'collection_stats.json'
with open(stats_file, 'w', encoding='utf-8') as f:
    json.dump({
        'timestamp': datetime.utcnow().isoformat(),
        'total_entities': len(entities),
        'spider_stats': collection_stats,
        'spider_factory_stats': factory.get_stats(),
    }, f, indent=2, default=str)

# Save regional breakdown
by_region = {}
by_country = {}
by_type = {}
for e in entities:
    r = e.get('region', 'Unknown')
    c = e.get('country', 'Unknown')
    t = e.get('site_type', 'Unknown')
    by_region[r] = by_region.get(r, 0) + 1
    by_country[c] = by_country.get(c, 0) + 1
    by_type[t] = by_type.get(t, 0) + 1

breakdown = {'by_region': by_region, 'by_country': by_country, 'by_type': by_type}
breakdown_file = DRIVE_DATA_DIR / 'data_breakdown.json'
with open(breakdown_file, 'w', encoding='utf-8') as f:
    json.dump(breakdown, f, indent=2)

print(f'')
print(f'Regional breakdown:')
for r, c in sorted(by_region.items(), key=lambda x: x[1], reverse=True):
    print(f'  {r:20s} {c:6d} entities')
print(f'')
print(f'Top 10 countries:')
for c, n in sorted(by_country.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f'  {c:20s} {n:6d} entities')

## Step 7: Generate Dashboard

In [ ]:
import json
import shutil
from commodity_os.dashboard.generator import DashboardGenerator

# Ensure category field for dashboard
for e in entities:
    if not e.get('category'):
        e['category'] = e.get('commodity_group', '')

# Generate dashboard
output_dir = Path(PROJECT_DIR) / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

dg = DashboardGenerator(str(output_dir))
json_data = dg.generate_json_data(entities, {})
html = dg.generate_html(json_data)
dg.save_files(json_data, html)

# Copy to docs for GitHub Pages
docs_dir = Path(PROJECT_DIR) / 'docs'
docs_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(output_dir / 'dashboard.html', docs_dir / 'index.html')
shutil.copy2(output_dir / 'dashboard.json', docs_dir / 'data.json')

# Also save to Google Drive
shutil.copy2(output_dir / 'dashboard.html', DRIVE_DATA_DIR / 'dashboard.html')
shutil.copy2(output_dir / 'dashboard.json', DRIVE_DATA_DIR / 'dashboard.json')

print(f'Dashboard generated with {len(entities)} entities')
print(f'  Output: {output_dir / "dashboard.html"}')
print(f'  Docs: {docs_dir / "index.html"}')
print(f'  Drive: {DRIVE_DATA_DIR / "dashboard.html"}')

## Step 8: Push to GitHub Pages

In [ ]:
import os
os.chdir(PROJECT_DIR)

# Stage and commit
!git add docs/ output/ commodity_os/
!git commit -m "Global collection: {0} entities from 2400+ spiders".format(len(entities))

# Push to GitHub
!git push origin {GITHUB_BRANCH}

print(f'')
print(f'Pushed to GitHub! Dashboard at:')
print(f'https://iamgokhu.github.io/Commodityapp01/')

## Step 9: View Results

In [ ]:
from IPython.display import HTML, display

# Display dashboard in Colab
dashboard_path = Path(PROJECT_DIR) / 'output' / 'dashboard.html'
if dashboard_path.exists():
    with open(dashboard_path, 'r', encoding='utf-8') as f:
        html_content = f.read()
    display(HTML(html_content))
else:
    print('Dashboard not found')

In [ ]:
# Print final summary
print(f'='*60)
print(f'FINAL SUMMARY')
print(f'='*60)
print(f'Total entities collected: {len(entities)}')
print(f'Regions covered: {len(by_region)}')
print(f'Countries covered: {len(by_country)}')
print(f'')
print(f'Files saved:')
print(f'  Google Drive: {DRIVE_DATA_DIR}')
print(f'  GitHub Pages: https://iamgokhu.github.io/Commodityapp01/')
print(f'')
print(f'Data saved to Google Drive:')
for f in sorted(DRIVE_DATA_DIR.iterdir()):
    size = f.stat().st_size / 1024
    print(f'  {f.name:30s} {size:8.1f} KB')
print(f'='*60)

## Step 10: Schedule Recurring Collection (Optional)

In [ ]:
# Auto-refresh loop - re-runs collection every N minutes
# Uncomment the code below to enable auto-collection

import time

RUN_INTERVAL_MINUTES = 30  # Change this to desired interval
MAX_RUNS = 10  # Max collection cycles

async def auto_collect_loop():
    """Run collection in a loop with Google Drive backup."""
    for run_num in range(1, MAX_RUNS + 1):
        print(f'\n{"="*60}')
        print(f'COLLECTION RUN #{run_num}')
        print(f'{"="*60}')

        # Run collection
        new_entities, stats = await factory.run_batch(batch_size=50, delay=3.0)

        # Merge with existing
        existing_file = DRIVE_DATA_DIR / 'all_entities.json'
        existing = []
        if existing_file.exists():
            with open(existing_file, encoding='utf-8') as f:
                existing = json.load(f)

        # Dedup by name+source
        seen = set()
        for e in existing:
            seen.add((e.get('name', ''), e.get('source', ''), e.get('country', '')))

        new_count = 0
        for e in new_entities:
            key = (e.get('name', ''), e.get('source', ''), e.get('country', ''))
            if key not in seen:
                existing.append(e)
                seen.add(key)
                new_count += 1

        # Save to Google Drive
        with open(existing_file, 'w', encoding='utf-8') as f:
            json.dump(existing, f, indent=2, ensure_ascii=False)

        # Regenerate dashboard
        for e in existing:
            if not e.get('category'):
                e['category'] = e.get('commodity_group', '')

        dg = DashboardGenerator(str(output_dir))
        json_data = dg.generate_json_data(existing, {})
        html = dg.generate_html(json_data)
        dg.save_files(json_data, html)
        shutil.copy2(output_dir / 'dashboard.html', docs_dir / 'index.html')
        shutil.copy2(output_dir / 'dashboard.json', docs_dir / 'data.json')

        # Push to GitHub
        os.chdir(PROJECT_DIR)
        os.system(f'git add docs/ output/')
        os.system(f'git commit -m "Auto collection run #{run_num}: {len(existing)} total entities"')
        os.system(f'git push origin {GITHUB_BRANCH}')

        print(f'Run #{run_num} complete: {new_count} new entities, {len(existing)} total')

        if run_num < MAX_RUNS:
            print(f'Next run in {RUN_INTERVAL_MINUTES} minutes...')
            await asyncio.sleep(RUN_INTERVAL_MINUTES * 60)

# Uncomment to start auto-collection:
# await auto_collect_loop()